# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Answer

Before testing individual signals, I inspected the distributions of several key fields used for content prioritization.

The dataset contains 30,000 content items. Search volume and content age are expected to be unevenly distributed, so I report both summary statistics and selected percentiles rather than relying only on the mean.

These distributions describe the observed dataset and are used to understand the range and concentration of the signals before testing their directional relationships.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

distribution_fields = [
    "search_volume",
    "days_since_last_update",
    "content_age_days",
    "word_count",
    "avg_position"
]

print("Dataset shape:", df.shape)

print("\nDistribution summary:")
print(
    df[distribution_fields].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).T
)

print("\nSkewness:")
print(
    df[distribution_fields]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
)

Dataset shape: (30000, 44)

Distribution summary:
                          count         mean          std   min     25%  \
search_volume           27532.0   158.882391  1518.270825   0.0     0.0   
days_since_last_update  30000.0    46.098300    42.078709   1.0    20.0   
content_age_days        30000.0   256.167800   132.707930  90.0   132.0   
word_count              22301.0  3107.760325  1452.382598   8.0  2413.0   
avg_position            30000.0    16.342380    15.216790   0.0     6.2   

                           50%     75%     90%     95%       99%      max  
search_volume             10.0    20.0   110.0   390.0  2900.000  74000.0  
days_since_last_update    20.0   104.0   104.0   104.0   106.000    373.0  
content_age_days         236.0   333.0   463.0   487.0   537.000    564.0  
word_count              2877.0  3666.0  5327.0  6173.0  7292.000   9546.0  
avg_position              10.8    22.3    36.8    48.2    69.901    245.0  

Skewness:
search_volume             26.016

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## 2. Answer

I tested three simple signals using observed performance measures.

**Signal 1 — Search volume:** I tested whether search volume is associated with total impressions. The expectation is directional: pages with greater search demand may have more opportunities to receive impressions.

**Signal 2 — Days since last update:** I tested whether freshness is associated with CTR. The expectation is directional: pages that have gone longer without an update may show lower CTR.

**Signal 3 — Content age:** I tested whether content age is associated with CTR. This checks whether older pages show a measurable difference in observed click-through behavior.

For each signal, the verdict is based on the direction and size of the measured relationship. A weak relationship is not treated as strong confirmation.

Possible verdicts are CONFIRMED, OPPOSITE, MIXED, or FALSE.

In [2]:
# -----------------------------
# Signal #1: Search volume vs impressions
# -----------------------------

signal_1 = df[
    ["search_volume", "impressions_90d"]
].dropna()

corr_1 = signal_1["search_volume"].corr(
    signal_1["impressions_90d"]
)

print("SIGNAL #1 — Search volume vs impressions")
print(f"Rows used: {len(signal_1)}")
print(f"Correlation: {corr_1:.3f}")

if corr_1 > 0.10:
    verdict_1 = "CONFIRMED"
elif corr_1 < -0.10:
    verdict_1 = "OPPOSITE"
else:
    verdict_1 = "MIXED"

print("Verdict:", verdict_1)


# -----------------------------
# Signal #2: Days since update vs CTR
# -----------------------------

signal_2 = df[
    ["days_since_last_update", "ctr"]
].dropna()

corr_2 = signal_2["days_since_last_update"].corr(
    signal_2["ctr"]
)

print("\nSIGNAL #2 — Days since last update vs CTR")
print(f"Rows used: {len(signal_2)}")
print(f"Correlation: {corr_2:.3f}")

if corr_2 < -0.10:
    verdict_2 = "CONFIRMED"
elif corr_2 > 0.10:
    verdict_2 = "OPPOSITE"
else:
    verdict_2 = "MIXED"

print("Verdict:", verdict_2)


# -----------------------------
# Signal #3: Content age vs CTR
# -----------------------------

signal_3 = df[
    ["content_age_days", "ctr"]
].dropna()

corr_3 = signal_3["content_age_days"].corr(
    signal_3["ctr"]
)

print("\nSIGNAL #3 — Content age vs CTR")
print(f"Rows used: {len(signal_3)}")
print(f"Correlation: {corr_3:.3f}")

if corr_3 < -0.10:
    verdict_3 = "CONFIRMED"
elif corr_3 > 0.10:
    verdict_3 = "OPPOSITE"
else:
    verdict_3 = "MIXED"

print("Verdict:", verdict_3)

SIGNAL #1 — Search volume vs impressions
Rows used: 27532
Correlation: 0.001
Verdict: MIXED

SIGNAL #2 — Days since last update vs CTR
Rows used: 30000
Correlation: -0.021
Verdict: MIXED

SIGNAL #3 — Content age vs CTR
Rows used: 30000
Correlation: 0.009
Verdict: MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. Answer

The baseline queue gives higher priority to content that has gone longer without an update.

I tested this assumption by comparing the observed decline-proxy rate for content below and above the median `days_since_last_update`.

The decline proxy is defined as `trend_direction == "down"`. It is used here only as an audit target, not as an input to the baseline score.

If the longer-unupdated group has a higher decline-proxy rate, the data supports the direction of the flag. If the difference is small or reverses, the assumption should be treated cautiously.

In [3]:
audit_df = df[
    ["days_since_last_update", "trend_direction"]
].copy()

audit_df["declining_proxy"] = (
    audit_df["trend_direction"] == "down"
).astype(int)

median_days = audit_df["days_since_last_update"].median()

audit_df["freshness_group"] = np.where(
    audit_df["days_since_last_update"] <= median_days,
    "At or below median days since update",
    "Above median days since update"
)

group_summary = (
    audit_df
    .groupby("freshness_group", observed=True)
    .agg(
        content_items=("declining_proxy", "size"),
        declining_items=("declining_proxy", "sum"),
        decline_proxy_rate=("declining_proxy", "mean"),
        median_days_since_update=("days_since_last_update", "median")
    )
)

print("Median days since last update:", median_days)

print("\nFlag-linked audit:")
print(group_summary)

rate_low = group_summary.loc[
    "At or below median days since update",
    "decline_proxy_rate"
]

rate_high = group_summary.loc[
    "Above median days since update",
    "decline_proxy_rate"
]

difference = rate_high - rate_low

print(f"\nDifference in decline-proxy rate: {difference:.3f}")

if difference > 0.05:
    flag_verdict = "CONFIRMED"
elif difference < -0.05:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

print("Flag-linked verdict:", flag_verdict)

Median days since last update: 20.0

Flag-linked audit:
                                      content_items  declining_items  \
freshness_group                                                        
Above median days since update                14134             7712   
At or below median days since update          15866             8550   

                                      decline_proxy_rate  \
freshness_group                                            
Above median days since update                  0.545635   
At or below median days since update            0.538888   

                                      median_days_since_update  
freshness_group                                                 
Above median days since update                           104.0  
At or below median days since update                      20.0  

Difference in decline-proxy rate: 0.007
Flag-linked verdict: MIXED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## 4. Answer

The audit shows which simple signals have measurable directional support in this dataset and which ones should be treated more cautiously.

For the content team, these signals should be treated as prioritization clues rather than automatic refresh decisions. The observed relationships do not establish causation, so pages surfaced by the flags still require human review and additional context.

In [4]:
print("Signal audit completed.")

print("\nSummary:")
print(f"Signal #1 verdict: {verdict_1}")
print(f"Signal #2 verdict: {verdict_2}")
print(f"Signal #3 verdict: {verdict_3}")
print(f"Flag-linked verdict: {flag_verdict}")

print("\nKey measured relationships:")
print(f"Search volume vs impressions correlation: {corr_1:.3f}")
print(f"Days since update vs CTR correlation: {corr_2:.3f}")
print(f"Content age vs CTR correlation: {corr_3:.3f}")
print(f"Flag-linked decline-rate difference: {difference:.3f}")

Signal audit completed.

Summary:
Signal #1 verdict: MIXED
Signal #2 verdict: MIXED
Signal #3 verdict: MIXED
Flag-linked verdict: MIXED

Key measured relationships:
Search volume vs impressions correlation: 0.001
Days since update vs CTR correlation: -0.021
Content age vs CTR correlation: 0.009
Flag-linked decline-rate difference: 0.007


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.